# Debug: OLMo-3 7B DPO steering sign-flip investigations

Three experiments to isolate the cause of the steering sign flip observed
in the main pipeline. Each experiment runs on OLMo-3 7B Instruct-DPO on
A100; Experiment 2 additionally loads OLMo-2 1B Instruct as an independent
judge. Peak VRAM ~18 GB (main 14 GB + judge 3 GB), fits comfortably on 40 GB.

**Experiment 1** — Truncation-window ablation: compare direction extracted
from first 15 completion tokens vs from the full completion. Tests whether
the opening-style confound causes the sign flip.

**Experiment 2** — Independent judge: re-judge the same outputs with OLMo-2
1B Instruct (NOT the model being steered). Tests whether self-judge bias
accounts for the flipped sign.

**Experiment 3** — Generation-time direction: extract the contrastive
direction from the model's OWN baseline generation activations (grouped by
judge label), not from teacher-forced completion activations. Tests whether
the direction needs to come from the generation regime rather than the
reading regime.

All experiments use N_TRAIN=100 direction stimuli, N_TEST=30 held-out
stimuli, alphas ∈ {-5, 0, +5}. Expected runtime ~60-90 min.

In [ ]:
# Clone repo and install deps. Do NOT -U numpy/scipy/sklearn/matplotlib/tqdm
# (Colab pins them for numba/tensorflow compat).
import os
if not os.path.exists('/content/demo'):
    !git clone https://github.com/elliott-leow/clin_syco_demo.git /content/demo
os.chdir('/content/demo')
!pip install -q "transformers>=4.57.0" accelerate

In [ ]:
import sys
sys.path.insert(0, '/content/demo/debug')
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from experiments import (
    set_seeds, cleanup,
    run_experiment_1_truncation,
    run_experiment_2_independent_judge,
    run_experiment_3_generation_time_direction,
)

set_seeds(42)

MAIN_MODEL_ID = 'allenai/Olmo-3-7B-Instruct-DPO'
JUDGE_INDEP_ID = 'allenai/OLMo-2-0425-1B-Instruct'
N_TRAIN = 100
N_TEST = 30
ALPHAS = [-5, 0, +5]

print(f'CUDA: {torch.cuda.is_available()}, GPU: '
      f'{torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(none)"}')

## Load main model (OLMo-3 7B Instruct-DPO)

In [ ]:
print(f'Loading {MAIN_MODEL_ID}...')
model = AutoModelForCausalLM.from_pretrained(
    MAIN_MODEL_ID, dtype=torch.bfloat16, low_cpu_mem_usage=True,
    attn_implementation='sdpa', device_map='auto')
model.eval()
tokenizer = AutoTokenizer.from_pretrained(MAIN_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

N_LAYERS = model.config.num_hidden_layers
STEP = max(1, N_LAYERS // 8)
LAYERS = sorted(set(list(range(0, N_LAYERS, STEP)) + [N_LAYERS - 1]))
MID_LAYER = LAYERS[len(LAYERS) // 2]
print(f'{N_LAYERS} layers, sampling {LAYERS}, MID={MID_LAYER}')
print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## Load stimuli

In [ ]:
stim = json.load(open('v2_clinical_cold.json'))
stim_train = stim[:N_TRAIN]
stim_test = stim[N_TRAIN:N_TRAIN + N_TEST]
print(f'Train: {len(stim_train)} items')
print(f'Test:  {len(stim_test)} items')

## Experiment 1 — Truncation window ablation

Same stimuli → extract direction two ways (first 15 tokens, full completion) → steer with each → blind-judge (self-judge for this cell; Experiment 2 will add the independent judge).

In [ ]:
import time
t0 = time.time()
exp1 = run_experiment_1_truncation(
    model, tokenizer, stim_train, stim_test, LAYERS, MID_LAYER,
    ALPHAS, judge_model=model, judge_tokenizer=tokenizer)
print(f'\nExperiment 1 runtime: {(time.time()-t0)/60:.1f} min')
cleanup()

## Experiment 2 — Independent judge

Load OLMo-2 1B Instruct as a second model (no overlap with the 7B DPO being steered). Re-judge the generations from Experiment 1. Compare per-config counts and agreement rate.

In [ ]:
print(f'Loading independent judge: {JUDGE_INDEP_ID}')
judge_model_indep = AutoModelForCausalLM.from_pretrained(
    JUDGE_INDEP_ID, dtype=torch.bfloat16, low_cpu_mem_usage=True,
    attn_implementation='sdpa', device_map='auto')
judge_model_indep.eval()
judge_tok_indep = AutoTokenizer.from_pretrained(JUDGE_INDEP_ID)
if judge_tok_indep.pad_token is None:
    judge_tok_indep.pad_token = judge_tok_indep.eos_token
print(f'VRAM after loading both models: {torch.cuda.memory_allocated()/1e9:.1f} GB')

t0 = time.time()
exp2 = run_experiment_2_independent_judge(
    exp1['rows'], exp1['verdicts'], judge_model_indep, judge_tok_indep)
print(f'\nExperiment 2 runtime: {(time.time()-t0)/60:.1f} min')

# Free the independent judge before Experiment 3
del judge_model_indep, judge_tok_indep
cleanup()

## Experiment 3 — Generation-time direction

Instead of extracting the direction from teacher-forced completion activations, extract it from the model's own generations on the TRAINING stimuli, grouped by the judge's post-hoc label. Then steer on held-out stimuli.

This is the most principled direction for a steering experiment because it uses generation-time representations, matching the regime where we intervene. If it also sign-flips, the flip is not an extraction-regime artifact.

In [ ]:
t0 = time.time()
exp3 = run_experiment_3_generation_time_direction(
    model, tokenizer, stim_train, stim_test, LAYERS, MID_LAYER,
    ALPHAS, judge_model=model, judge_tokenizer=tokenizer)
print(f'\nExperiment 3 runtime: {(time.time()-t0)/60:.1f} min')
cleanup()

## Summary

Collate all three experiments' findings to answer: which of the three hypothesized causes (truncation, self-judge bias, regime mismatch) best accounts for the sign flip?

In [ ]:
os.makedirs('debug/results', exist_ok=True)
out = {'exp1': exp1, 'exp2': exp2, 'exp3': exp3,
       'config': {'main_model': MAIN_MODEL_ID,
                   'judge_independent': JUDGE_INDEP_ID,
                   'n_train': N_TRAIN, 'n_test': N_TEST,
                   'alphas': ALPHAS, 'layers': LAYERS,
                   'mid_layer': MID_LAYER}}
json.dump(out, open('debug/results/debug_results.json', 'w'),
          indent=2, default=str)
print('Saved debug/results/debug_results.json')

# Quick sign-check summary:
def sign_check(summary, alpha_labels):
    """Return 'CORRECT' if subtracting-direction alpha>0 lowers syc rate."""
    base_syc = summary.get('baseline', {}).get('sycophantic', 0)
    plus_syc = summary.get(alpha_labels['plus'], {}).get('sycophantic', base_syc)
    minus_syc = summary.get(alpha_labels['minus'], {}).get('sycophantic', base_syc)
    plus_ok = plus_syc < base_syc  # subtract direction → less syc
    minus_ok = minus_syc > base_syc  # add direction → more syc
    return ('CORRECT' if (plus_ok and minus_ok) else
            'FLIPPED' if (plus_syc > base_syc and minus_syc < base_syc) else
            'INCONCLUSIVE')

print('\n' + '='*70)
print('SIGN-CHECK SUMMARY')
print('='*70)
print('Exp 1 — 15-token direction (self-judge):',
      sign_check(exp1['summary'], {'plus': '15tok_a+5', 'minus': '15tok_a-5'}))
print('Exp 1 — full-completion direction (self-judge):',
      sign_check(exp1['summary'], {'plus': 'full_a+5', 'minus': 'full_a-5'}))
print('Exp 2 — 15-tok direction (independent judge):',
      sign_check(exp2['summary_independent'], {'plus': '15tok_a+5', 'minus': '15tok_a-5'}))
print('Exp 2 — full direction (independent judge):',
      sign_check(exp2['summary_independent'], {'plus': 'full_a+5', 'minus': 'full_a-5'}))
print('Exp 3 — generation-time direction (self-judge):',
      sign_check(exp3['summary'], {'plus': 'gen_a+5', 'minus': 'gen_a-5'}))

print('\nRead the full per-config counts above to see rate changes.')